# Tests: `fasterai.misc.cpu_optimizer` (source `nbs/misc/cpu_optimizer.ipynb`)

In [ ]:
from fastcore.test import *
from fasterai.misc.cpu_optimizer import *

In [ ]:
from fastcore.test import *
import torch, torch.nn as nn

# optimize_for_cpu with trace backend
_m = nn.Sequential(nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(16, 10))
_x = torch.randn(1, 3, 8, 8)
_traced = optimize_for_cpu(_m, _x, backend="trace")
_out = _traced(_x.to(memory_format=torch.channels_last))
test_eq(_out.shape, (1, 10))
assert torch.isfinite(_out).all()

# Invalid backend raises ValueError
with ExceptionExpected(ValueError): optimize_for_cpu(_m, _x, backend="bad")

# Deprecated function emits warning
import warnings
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    accelerate_model_for_cpu(nn.Sequential(nn.Conv2d(3, 16, 3), nn.ReLU()), torch.randn(1, 3, 8, 8))
    dep_warnings = [x for x in w if issubclass(x.category, DeprecationWarning)]
    assert len(dep_warnings) >= 1, f"Expected DeprecationWarning, got {[x.category for x in w]}"